# MultiVI analysis (Gladstone, NK and CD8 T, PID 9 and 12)

This notebook runs MultiVI on paired snRNA-seq and snATAC-seq data and ranks genes for the MultiVI comparison in Fig. 1c.

**Input** comes from `export_multiome_for_MultiVI.Rmd`, with one folder per PID: `MultiVI_input/PID_<pid>/`. Each folder contains:

- `rna_counts.mtx`: genes x cells, raw RNA counts
- `rna_genes.tsv`
- `atac_counts.mtx`: peaks x cells, ArchR PeakMatrix counts
- `atac_peaks.tsv`
- `cells.csv`: cell, PID, cell_type

**Steps, for each PID:**

1. Build a MuData with two modalities, `rna` and `atac`, and drop rarely detected features. All genes are kept; peaks are kept if detected in at least 1% of cells (as in the MultiVI tutorial).
2. Train scvi-tools `MULTIVI` on the paired data.
3. For every gene and cell type, compute the mean MultiVI-normalized expression (`px_scale`, averaged over the cells of that type) in three ways:
   - **combined**: each cell is encoded from both RNA and ATAC (the joint representation).
   - **RNA**: each cell is encoded from RNA only, with the ATAC input masked.
   - **ATAC**: each cell is encoded from ATAC only, with the RNA input masked.

   Masking a modality is how MultiVI handles unpaired cells: the latent representation then comes only from the encoder of the observed modality.
4. Rank genes within each cell type (rank 1 = highest). Then write two outputs:
   - the top 10,000 genes by combined score, as input for Incytr;
   - the three ranks for the genes of interest, with the same columns as `MultiVI_3score_comparison.csv`, for `STTT_RankingPlot.Rmd`.

**Environment** (one-time setup in Anaconda Prompt). RTX 50-series GPUs need a PyTorch build for CUDA 12.8 or later.
```
conda create -n multivi python=3.12 -y
conda activate multivi
pip install torch --index-url https://download.pytorch.org/whl/cu128
pip install scvi-tools mudata scanpy ipykernel
python -m ipykernel install --user --name multivi --display-name "Python (multivi)"
```
Then choose the kernel **Python (multivi)** in Jupyter or VS Code.

This was tested with scvi-tools 1.5.1 and mudata 0.4.1. Since scvi-tools 1.4, MULTIVI requires MuData input.

## Packages

In [ ]:
import json
import math
import platform
import time
from importlib.metadata import version
from pathlib import Path

import anndata as ad
import mudata as md
import numpy as np
import pandas as pd
import scipy.io as sio
import scipy.sparse as sp
import scvi
import torch

## Settings

To check the setup first, set `TEST = True`. This runs about 600 cells and 5 epochs, and writes to `MultiVI_output_test`.

In [ ]:
BASE = Path(r"C:\Users\chang\OneDrive\Documents\Research_2025\Gladstone\MultiVI")

PIDS = [9, 12]
CELLTYPES = {"NK": "NK", "CD8 T": "CD8_T"}  # label in cells.csv -> label in output files
N_TOP = 10000                                # top-N genes passed to Incytr
MIN_CELLS_GENE = 0                           # 0 = keep all genes (same 33,258 genes as scVI / Incytr)
MIN_FRAC_PEAK = 0.01                         # keep peaks detected in >= 1% of cells (MultiVI tutorial)
SEED = 0

# genes of interest (same lists as in MOFA2_Gladstone.Rmd / MultiVI_Gladstone.Rmd)
NK_PANEL = [
    "CD57", "CD14", "CD25", "CD8", "CCR7", "PD1", "CD7", "TIM-3", "NKG2D", "TIGIT",
    "NKG2C", "CD27", "NKp30", "CXCR6", "CD45RO", "CD69", "CD49a", "CXCR5", "CD127",
    "KIR2DL1", "KIR3DL1", "DNAM-1", "NKG2A", "CD3", "CD38", "CD4", "CD56",
    "HLA-DR", "CD16", "IFITM1", "IFIT1", "AIM2", "IFIT3", "SLFN11", "MX2", "pIRF3",
    "MX1", "RIGI", "BRD4", "OAS1",
]
NK_SIGNAL = ["PTPRC", "HLA-DMB", "HLA-DMA", "HLA-DPB1", "HLA-G"]
CD8_SIGNAL = [
    "SLFN11", "AIM2", "CCR7", "IFITM1", "CD7", "MX1", "TIGIT", "MX2", "CD38", "IFIT3",
    "CXCR5", "BRD4", "CD14", "CD69", "MAPK14", "HNRNPD", "HNRNPH1", "YBX1", "FOS",
    "SYNCRIP", "NFATC3", "RARA", "ITK", "NR3C1", "DDX5", "NPM1", "SLA2", "CD4",
]
GENES_OF_INTEREST = {"NK": list(dict.fromkeys(NK_PANEL + NK_SIGNAL)), "CD8_T": CD8_SIGNAL}

# ---- run options -------------------------------------------------------------
TEST = False          # True = quick check on a small subset (~1 min); set False for the real run
MAX_EPOCHS = 500      # upper limit; training stops early when the validation loss stops improving
ACCELERATOR = "auto"  # "auto" uses the GPU if PyTorch can see it; "cpu" forces CPU

## Helper functions: data

In [ ]:
def load_pid(in_dir: Path, test: bool = False) -> tuple[md.MuData, dict]:
    """Read one PID's exported matrices into a MuData with modalities "rna" and "atac"."""
    rna = sio.mmread(in_dir / "rna_counts.mtx").tocsr()     # genes x cells
    atac = sio.mmread(in_dir / "atac_counts.mtx").tocsr()   # peaks x cells
    genes = [g.strip() for g in (in_dir / "rna_genes.tsv").read_text().splitlines() if g.strip()]
    peaks = pd.read_csv(in_dir / "atac_peaks.tsv", sep="\t")["peak"].astype(str).str.strip().tolist()
    cells = pd.read_csv(in_dir / "cells.csv")

    assert rna.shape == (len(genes), len(cells)), f"RNA matrix {rna.shape} vs {len(genes)} genes, {len(cells)} cells"
    assert atac.shape == (len(peaks), len(cells)), f"ATAC matrix {atac.shape} vs {len(peaks)} peaks, {len(cells)} cells"
    assert len(set(genes)) == len(genes) and len(set(peaks)) == len(peaks)

    obs = cells.set_index("cell")
    obs.index = obs.index.astype(str)
    obs["cell_type"] = obs["cell_type"].astype(str)
    a_rna = ad.AnnData(X=rna.T.tocsr().astype(np.float32), obs=obs.copy(),
                       var=pd.DataFrame(index=pd.Index(genes, name="gene")))
    a_atac = ad.AnnData(X=atac.T.tocsr().astype(np.float32), obs=obs.copy(),
                        var=pd.DataFrame(index=pd.Index(peaks, name="peak")))

    if test:  # small subset for a quick end-to-end check
        rng = np.random.default_rng(SEED)
        keep_cells = np.sort(np.concatenate([
            rng.choice(np.where(obs["cell_type"].values == ct)[0],
                       size=min(300, int((obs["cell_type"] == ct).sum())), replace=False)
            for ct in CELLTYPES
        ]))
        a_rna = a_rna[keep_cells, :3000].copy()
        a_atac = a_atac[keep_cells, :10000].copy()

    # filter rarely detected features
    det_rna = np.asarray((a_rna.X > 0).sum(axis=0)).ravel()
    det_atac = np.asarray((a_atac.X > 0).sum(axis=0)).ravel()
    info = {
        "n_cells": int(a_rna.n_obs),
        "n_cells_by_type": a_rna.obs["cell_type"].value_counts().to_dict(),
        "n_genes_input": int(a_rna.n_vars),
        "n_peaks_input": int(a_atac.n_vars),
    }
    a_rna = a_rna[:, det_rna >= MIN_CELLS_GENE].copy()
    a_atac = a_atac[:, det_atac >= math.ceil(MIN_FRAC_PEAK * a_atac.n_obs)].copy()
    info["n_genes_kept"], info["n_peaks_kept"] = int(a_rna.n_vars), int(a_atac.n_vars)

    mdata = md.MuData({"rna": a_rna, "atac": a_atac})
    mdata.obs["cell_type"] = a_rna.obs["cell_type"].values
    return mdata, info


def masked_copy(mdata: md.MuData, mask_mod: str) -> md.MuData:
    """Copy of mdata with modality `mask_mod` set to zero (MultiVI then treats it as unmeasured)."""
    mods = {}
    for k, a in mdata.mod.items():
        X = sp.csr_matrix(a.X.shape, dtype=np.float32) if k == mask_mod else a.X.copy()
        mods[k] = ad.AnnData(X=X, obs=a.obs.copy(), var=a.var.copy())
    out = md.MuData(mods)
    out.obs["cell_type"] = mdata.obs["cell_type"].values
    out.obs["_indices"] = np.arange(out.n_obs)   # required by MULTIVI's MuData setup
    return out

## Helper functions: scores

In [ ]:
def mean_expression_by_celltype(model, mdata: md.MuData) -> pd.DataFrame:
    """Mean MultiVI-normalized expression (px_scale) of every gene in each cell type."""
    out = {}
    for ct, ct_label in CELLTYPES.items():
        idx = np.where(mdata.obs["cell_type"].values == ct)[0]
        expr = model.get_normalized_expression(
            adata=mdata, indices=idx, use_z_mean=True, return_numpy=True, batch_size=256
        )
        out[ct_label] = np.asarray(expr).mean(axis=0)
    return pd.DataFrame(out, index=mdata.mod["rna"].var_names)


def rank_desc(s: pd.Series) -> pd.Series:
    return s.rank(ascending=False, method="first", na_option="keep")

## Check the GPU and set up the output folder

In [ ]:
in_root = BASE / "MultiVI_input"
out_root = BASE / ("MultiVI_output_test" if TEST else "MultiVI_output")
out_root.mkdir(parents=True, exist_ok=True)
max_epochs = 5 if TEST else MAX_EPOCHS

scvi.settings.seed = SEED
torch.set_float32_matmul_precision("high")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)} | torch {torch.__version__} | CUDA {torch.version.cuda}")
else:
    print(f"No CUDA GPU visible to torch {torch.__version__} - training on CPU (slow).")

comparison = []
run_info = {
    "settings": {"N_TOP": N_TOP, "MIN_CELLS_GENE": MIN_CELLS_GENE, "MIN_FRAC_PEAK": MIN_FRAC_PEAK,
                 "SEED": SEED, "max_epochs": max_epochs, "test": TEST},
    "versions": {p: version(p) for p in ["scvi-tools", "torch", "anndata", "mudata", "numpy", "scipy", "pandas"]},
    "python": platform.python_version(),
    "device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu",
    "pids": {},
}

## Train MultiVI and compute the gene scores for each PID

In [ ]:
for pid in PIDS:
    t0 = time.time()
    print(f"\n===== PID {pid} =====")
    mdata, info = load_pid(in_root / f"PID_{pid}", test=TEST)
    print(json.dumps(info, indent=2))
    n_genes, n_regions = mdata.mod["rna"].n_vars, mdata.mod["atac"].n_vars

    # --- train MultiVI -------------------------------------------------------
    scvi.model.MULTIVI.setup_mudata(mdata, modalities={"rna_layer": "rna", "atac_layer": "atac"})
    model = scvi.model.MULTIVI(mdata, n_genes=n_genes, n_regions=n_regions)
    model.train(max_epochs=max_epochs, accelerator=ACCELERATOR, early_stopping=True)
    model.save(str(out_root / f"MultiVI_model_PID{pid}"), overwrite=True)
    hist = pd.concat({k: v.iloc[:, 0] for k, v in model.history.items()
                      if k in ("elbo_train", "elbo_validation")}, axis=1)
    hist.to_csv(out_root / f"MultiVI_training_history_PID{pid}.csv")
    info["epochs_trained"] = int(len(hist))

    # --- scores: combined, RNA-only, ATAC-only ------------------------------
    scores = {
        "combined": mean_expression_by_celltype(model, mdata),
        "rna": mean_expression_by_celltype(model, masked_copy(mdata, "atac")),    # RNA only
        "atac": mean_expression_by_celltype(model, masked_copy(mdata, "rna")),    # ATAC only
    }

    gene_tables = []
    for ct_label in CELLTYPES.values():
        df = pd.DataFrame({f"score_{k}": v[ct_label] for k, v in scores.items()})
        for k in scores:
            df[f"rank_{k}"] = rank_desc(df[f"score_{k}"]).astype("Int64")
        df.index.name = "gene"
        df["cell_type"], df["PID"] = ct_label, pid
        gene_tables.append(df.reset_index())

        # top-N genes (combined) for Incytr
        top = df.sort_values("rank_combined").head(N_TOP).reset_index()
        top[["gene", "rank_combined", "score_combined", "score_rna", "score_atac"]].to_csv(
            out_root / f"MultiVI_top{N_TOP}_PID{pid}_{ct_label}.csv", index=False)

        # ranks of the genes of interest (genes filtered out -> NA)
        goi = GENES_OF_INTEREST[ct_label]
        cmp = pd.DataFrame({"gene": goi})
        for k in ("rna", "atac", "combined"):
            cmp[f"rank_{k}"] = df[f"rank_{k}"].reindex(goi).values
        cmp["cell_type"], cmp["PID"] = ct_label, pid
        comparison.append(cmp.sort_values("rank_combined", na_position="last"))

    pd.concat(gene_tables).to_csv(out_root / f"MultiVI_gene_scores_PID{pid}.csv", index=False)
    info["n_genes_ranked"], info["n_peaks_model"] = n_genes, n_regions
    info["minutes"] = round((time.time() - t0) / 60, 1)
    run_info["pids"][str(pid)] = info
    print(f"PID {pid} done in {info['minutes']} min ({info['epochs_trained']} epochs)")

## Save the comparison table and run information

In [ ]:
comp = pd.concat(comparison, ignore_index=True)
comp.index = comp.index + 1
comp.to_csv(out_root / "MultiVI_3score_comparison.csv")   # row index -> column "X" in R read.csv
(out_root / "run_info.json").write_text(json.dumps(run_info, indent=2))
print(f"Outputs written to {out_root}")

## Ranks of the CD8 T target genes shown in Fig. 1c

In [ ]:
comp[(comp["cell_type"] == "CD8_T") &
     comp["gene"].isin(["CD4", "MX1", "MX2", "IFIT3", "SLFN11", "IFITM1", "BRD4", "AIM2"])]

## Training curves (check that the validation loss has levelled off)

In [ ]:
for pid in PIDS:
    h = pd.read_csv(out_root / f"MultiVI_training_history_PID{pid}.csv", index_col=0)
    ax = h.plot(title=f"PID {pid}", xlabel="epoch", ylabel="ELBO", figsize=(5, 3))